# Μάθημα 11 - Πρωτόκολλο Πράκτορα προς Πράκτορα (A2A)


## Εγκατάσταση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Τι είναι το Πρωτόκολλο A2A;

Το **Agent-to-Agent (A2A) πρωτόκολλο** είναι ένα ανοιχτό πρότυπο που επιτρέπει στους πράκτορες AI να επικοινωνούν,
να ανακαλύπτουν ο ένας τον άλλο και να συνεργάζονται — ακόμη και όταν έχουν δημιουργηθεί σε διαφορετικά πλαίσια ή φιλοξενούνται
από διαφορετικές υπηρεσίες.

Κύριες έννοιες:

- **Ανακάλυψη** – Οι πράκτορες δημοσιεύουν μια *Κάρτα Πράκτορα* που περιγράφει τις δυνατότητές τους, καθιστώντας
  εύκολο για άλλους πράκτορες (ή διευθυντές ορχήστρας) να βρουν τον κατάλληλο ειδικό για μια εργασία.
- **Περιαγωγή Μηνυμάτων** – Οι πράκτορες ανταλλάσσουν δομημένα μηνύματα μέσω ενός κοινού πρωτοκόλλου, ώστε ένα
  αίτημα από έναν πράκτορα να μπορεί να γίνει κατανοητό και να εκπληρωθεί από έναν άλλο ανεξάρτητα από την εσωτερική
  υλοποίηση του.
- **Κύκλος Ζωής Εργασίας** – Το A2A ορίζει καταστάσεις όπως *υποβλήθηκε*, *σε εξέλιξη*, *ολοκληρώθηκε*, και
  *απέτυχε*, δίνοντας στον διευθυντή ορχήστρας πλήρη ορατότητα για το πώς εξελίσσεται μια ανατεθειμένη εργασία.

Σε αυτό το μάθημα προσομοιώνουμε τη συνεργασία με το στυλ A2A συνδέοντας τρεις εξειδικευμένους ταξιδιωτικούς πράκτορες
σε μια ροή εργασίας όπου κάθε πράκτορας συνεισφέρει την εξειδίκευσή του και μεταβιβάζει τα αποτελέσματα στον επόμενο.


## Δημιουργία Εξειδικευμένων Ταξιδιωτικών Πρακτόρων


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Multi-Agent Collaboration via Workflow

We connect the three agents into a sequential workflow that mirrors A2A message passing:

1. **CurrencyExchangeAgent** receives the user request and produces currency guidance.
2. **ActivityPlannerAgent** receives the enriched context and adds activity recommendations.
3. **TravelManagerAgent** synthesizes both inputs into a final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Κατανόηση του A2A στην Παραγωγή

Σε ένα περιβάλλον παραγωγής το πρωτόκολλο A2A ξεκλειδώνει ισχυρά σενάρια διασύνδεσης υπηρεσιών:

| Ικανότητα | Περιγραφή |
|---|---|
| **Διαλειτουργικότητα μεταξύ πλαισίων εργασίας** | Ένας πράκτορας που δημιουργήθηκε με ένα πλαίσιο εργασίας μπορεί να αναθέσει εργασίες σε έναν πράκτορα που δημιουργήθηκε με οποιοδήποτε άλλο πλαίσιο συμβατό με A2A, επιτρέποντας πραγματική διαλειτουργικότητα μεταξύ οργανισμών. |
| **Όρια υπηρεσιών** | Οι πράκτορες μπορούν να ζουν σε ξεχωριστές μικροϋπηρεσίες, περιοχές cloud ή ακόμα και σε διαφορετικούς οργανισμούς ενώ συνεργάζονται άψογα. |
| **Δυναμική ανακάλυψη** | Ένας οργανωτής μπορεί να ρωτήσει ένα μητρώο Κάρτας Πράκτορα κατά το χρόνο εκτέλεσης για να βρει τον κατάλληλο ειδικό για μια δεδομένη υποεργασία. |
| **Ροή και ειδοποιήσεις push** | Το A2A υποστηρίζει Server-Sent Events (SSE) για ενημερώσεις προόδου σε πραγματικό χρόνο και ειδοποιήσεις push για εργασίες που διαρκούν πολύ. |

Η ροή εργασίας που δημιουργήσαμε παραπάνω είναι μια απλοποιημένη, ενδο-διαδικαστική έκδοση αυτού του προτύπου. Σε μια πραγματική
ανάπτυξη κάθε πράκτορας θα έδειχνε ένα endpoint HTTP, θα δημοσίευε μια Κάρτα Πράκτορα και θα επικοινωνούσε
μέσω του πρωτοκόλλου A2A JSON-RPC.


## Περίληψη

Σε αυτό το μάθημα μάθατε:

1. **Τι είναι το πρωτόκολλο A2A** — ένα ανοιχτό πρότυπο για την ανακάλυψη, την ανταλλαγή μηνυμάτων,
   και τη διαχείριση εργασιών μεταξύ πρακτόρων.
2. **Πώς να δημιουργήσετε εξειδικευμένους πράκτορες** — έναν πράκτορα ανταλλαγής νομισμάτων, έναν πράκτορα προγραμματισμού δραστηριοτήτων,
   και έναν συντονιστή διαχείρισης ταξιδιών.
3. **Πώς να συνδέσετε πράκτορες σε μια ροή εργασίας** — χρησιμοποιώντας το `WorkflowBuilder` για να μοντελοποιήσετε τη διαδοχική
   ανταλλαγή μηνυμάτων μεταξύ των πρακτόρων.
4. **Πώς λειτουργεί το A2A στην παραγωγή** — επιτρέποντας συνεργασία μεταξύ πλαισίων και υπηρεσιών
   με δυναμική ανακάλυψη και ροή ενημερώσεων.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
